# AI5707 — Lab 5: Maximum Likelihood Parameter Estimation

## Objective
Estimate parameters using Maximum Likelihood Estimation (MLE) on the previous-assignment data and on real data.

We assume Gaussian observation noise:

$$t_n=\mathbf{w}^{T}\boldsymbol{\phi}(x_n)+\epsilon_n,\qquad \epsilon_n\sim\mathcal{N}(0,\beta^{-1}).$$

The MLE estimate of the regression parameters is

$$\boxed{\mathbf{w}_{ML}=(\Phi^{T}\Phi)^{-1}\Phi^{T}\mathbf{t}}$$

and the MLE noise variance is

$$\boxed{\sigma^2_{ML}=\frac{1}{N}\sum_{n=1}^{N}(t_n-\hat t_n)^2}$$

with precision $\beta_{ML}=1/\sigma^2_{ML}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
np.set_printoptions(precision=6, suppress=True)

def polynomial_design(x, degree):
    x = np.asarray(x).ravel()
    return np.column_stack([x**j for j in range(degree + 1)])

def gaussian_mle(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    w, _, rank, s = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ w
    residuals = y - y_hat
    sse = float(residuals @ residuals)
    N = len(y)
    sigma2_ml = sse / N
    beta_ml = np.inf if sigma2_ml == 0 else 1.0 / sigma2_ml
    return w, y_hat, residuals, sse, sigma2_ml, beta_ml, rank, s

## 1. Previous-assignment data

The data source is copied inside the Lab 5 folder, so this notebook is self-contained.

The local file is `data/noisy_16.txt`.

In [ ]:
data_path = Path('data/noisy_16.txt')
if not data_path.exists():
    raise FileNotFoundError('Expected data/noisy_16.txt. Run this notebook from the Lab 5 folder.')

data = np.loadtxt(data_path)
x, t = data[:, 0], data[:, 1]
print('Dataset shape:', data.shape)
print(data[:5])

In [ ]:
degree = 3
Phi = polynomial_design(x, degree)
w_ml, t_hat, residuals, sse, sigma2_ml, beta_ml, rank, s = gaussian_mle(Phi, t)

print('MLE weights:', w_ml)
print('SSE:', sse)
print('MLE noise variance σ²:', sigma2_ml)
print('MLE precision β:', beta_ml)
print('Design-matrix rank:', rank)

x_grid = np.linspace(x.min(), x.max(), 400)
plt.figure(figsize=(9, 5))
plt.scatter(x, t, s=18, label='Observed data')
plt.plot(x_grid, polynomial_design(x_grid, degree) @ w_ml, linewidth=2, label='MLE polynomial')
plt.xlabel('x'); plt.ylabel('t'); plt.title('Previous-assignment data: MLE polynomial regression')
plt.legend(); plt.grid(alpha=0.25); plt.show()

plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1); plt.scatter(x, residuals, s=18)
plt.xlabel('x'); plt.ylabel('Residual'); plt.title('Residuals of the MLE fit')
plt.grid(alpha=0.25); plt.show()

## 2. MLE derivation

For independent Gaussian errors,

$$p(\mathbf{t}\mid\Phi,\mathbf{w},\beta)=\prod_{n=1}^{N}\mathcal{N}\left(t_n\mid\mathbf{w}^{T}\boldsymbol{\phi}_n,\beta^{-1}\right).$$

Taking the logarithm,

$$\ln p=\frac{N}{2}\ln\beta-\frac{N}{2}\ln(2\pi)-\frac{\beta}{2}\sum_{n=1}^{N}\left(t_n-\mathbf{w}^{T}\boldsymbol{\phi}_n\right)^2.$$

Setting $\partial\ln p/\partial\mathbf{w}=0$ gives

$$\Phi^{T}\Phi\mathbf{w}_{ML}=\Phi^{T}\mathbf{t}.$$

Therefore,

$$\boxed{\mathbf{w}_{ML}=(\Phi^{T}\Phi)^{-1}\Phi^{T}\mathbf{t}}.$$

Differentiating with respect to $\beta$ gives

$$\boxed{\beta_{ML}=\frac{N}{\sum_{n=1}^{N}(t_n-\hat t_n)^2}}.$$

Hence,

$$\boxed{\sigma^2_{ML}=\frac{\mathrm{SSE}}{N}}.$$

The implementation uses `np.linalg.lstsq` instead of explicitly computing the inverse because it is numerically more stable.

In [ ]:
w_normal = np.linalg.solve(Phi.T @ Phi, Phi.T @ t)
print('Normal-equation weights:', w_normal)
print('Maximum coefficient difference:', np.max(np.abs(w_normal - w_ml)))

## 3. Real data — Diabetes dataset

We estimate a linear Gaussian regression model:

$$\mathbf{y}=X\mathbf{w}+\boldsymbol{\epsilon},\qquad \boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0},\sigma^2I).$$

An intercept column is added explicitly.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

dataset = load_diabetes()
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

X_train = np.column_stack([np.ones(len(X_train)), X_train])
X_test = np.column_stack([np.ones(len(X_test)), X_test])

w_real, y_train_hat, residuals_real, sse_real, sigma2_real, beta_real, rank_real, _ = gaussian_mle(X_train, y_train)
y_test_hat = X_test @ w_real

print('MLE coefficients [intercept, features]:')
print(w_real)
print('MLE noise variance σ²:', sigma2_real)
print('MLE precision β:', beta_real)
print('Train RMSE:', np.sqrt(mean_squared_error(y_train, y_train_hat)))
print('Test RMSE:', np.sqrt(mean_squared_error(y_test, y_test_hat)))
print('Test R²:', r2_score(y_test, y_test_hat))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_test_hat, alpha=0.75)
limits = [min(y_test.min(), y_test_hat.min()), max(y_test.max(), y_test_hat.max())]
plt.plot(limits, limits, linewidth=2)
plt.xlabel('Actual target'); plt.ylabel('Predicted target'); plt.title('Real data: MLE regression')
plt.grid(alpha=0.25); plt.show()

plt.figure(figsize=(8, 4))
plt.axhline(0, linewidth=1); plt.scatter(y_test_hat, y_test - y_test_hat, alpha=0.75)
plt.xlabel('Predicted target'); plt.ylabel('Residual'); plt.title('Real-data test residuals')
plt.grid(alpha=0.25); plt.show()

## 4. Conclusions

- Under Gaussian noise, MLE of the regression weights is equivalent to least squares.
- The MLE noise variance is $\mathrm{SSE}/N$.
- The unbiased variance estimator uses a different denominator, $N-p$.
- `np.linalg.lstsq` is numerically safer than explicitly calculating $(X^TX)^{-1}$.

### Computational complexity
For $N$ samples and $d$ parameters: forming $X^TX$ costs $O(Nd^2)$, solving the system costs approximately $O(d^3)$, prediction costs $O(Nd)$, and storing $X$ costs $O(Nd)$.
